In [0]:

CREATE OR REPLACE PROCEDURE dbw_healthcare_de_dev.etl.usp_claim_file(
    IN p_file_id STRING,
    IN p_adf_run_id STRING,
    IN p_bronze_file_exists BOOLEAN
)
LANGUAGE SQL
SQL SECURITY INVOKER
AS
BEGIN
    DECLARE v_rows_claimed BIGINT DEFAULT 0;

    IF p_bronze_file_exists = FALSE THEN

        UPDATE dbw_healthcare_de_dev.etl.file_control
        SET
            pipeline_status = 'IN_PROGRESS',
            processing_stage = 'BRONZE',
            adf_run_id = p_adf_run_id,
            processing_start_ts = CURRENT_TIMESTAMP(),
            last_updated_ts = CURRENT_TIMESTAMP(),
            error_message = NULL
        WHERE file_id = p_file_id
          AND pipeline_status = 'PENDING'
          AND processing_stage = 'REGISTERED'
          AND is_enabled = TRUE;

        GET DIAGNOSTICS v_rows_claimed = ROW_COUNT;

    END IF;

    SELECT
        CASE WHEN v_rows_claimed = 1
             THEN TRUE ELSE FALSE END AS claim_success,
        v_rows_claimed AS rows_claimed;
END;
